# Modelagem de Sobrevida — LUAD

Este notebook realiza a modelagem de sobrevida para o subtipo LUAD.

A base usada aqui já foi tratada no notebook anterior de tratamento LUAD.

Modelos avaliados:
1. Cox Proportional Hazards
2. Weibull AFT
3. Random Survival Forest
4. Gradient Boosting Survival Analysis
5. DeepSurv
6. DeepHit

Métrica principal:
- C-index

A validação será feita com 5 folds usando `StratifiedKFold`, mantendo a proporção de eventos em cada fold.

In [2]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler

from lifelines import CoxPHFitter, WeibullAFTFitter
from lifelines.utils import concordance_index

from sksurv.util import Surv
from sksurv.ensemble import RandomSurvivalForest, GradientBoostingSurvivalAnalysis
from sksurv.metrics import concordance_index_censored

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

In [3]:
try:
    import torch
    import torchtuples as tt

    from pycox.models import CoxPH, DeepHitSingle
    from pycox.evaluation import EvalSurv

    DEEP_OK = True
    print("Bibliotecas de deep learning carregadas com sucesso.")

except Exception as e:
    DEEP_OK = False
    print("Não foi possível carregar pycox/torch.")
    print("Erro:", e)

Bibliotecas de deep learning carregadas com sucesso.


In [4]:
COHORT = "LUAD"

PROJECT_DIR = Path("..").resolve()

DATA_PATH = PROJECT_DIR / "data" / "processed" / "luad" / "luad_modeling_dataset.csv"

RESULTS_DIR = PROJECT_DIR / "reports" / "results" / "luad"
FIGURES_DIR = PROJECT_DIR / "reports" / "figures" / "luad"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

TIME_COL = "surv__OS.time"
EVENT_COL = "surv__OS"

SEED = 42
N_SPLITS = 5

In [5]:
print("Carregando base final de modelagem LUAD...")
print(DATA_PATH)

df = pd.read_csv(DATA_PATH, sep=";")

print("Shape original:", df.shape)
display(df.head())

Carregando base final de modelagem LUAD...
/workspaces/An-lise-de-Sobrevida/data/processed/luad/luad_modeling_dataset.csv
Shape original: (502, 92)


,patient_id,surv__OS.time,surv__OS,clin__age_at_index.demographic,clin__age_at_earliest_diagnosis_in_years.diagnoses.xena_derived,clin__pack_years_smoked.exposures,clin__cigarettes_per_day.exposures,clin__age_at_index.demographic_missing,clin__age_at_earliest_diagnosis_in_years.diagnoses.xena_derived_missing,clin__pack_years_smoked.exposures_missing,clin__cigarettes_per_day.exposures_missing,clin__gender_bin,clin__prior_malignancy_bin,clin__prior_treatment_bin,clin__stage_group_I,clin__stage_group_II,clin__stage_group_III,clin__stage_group_IV,clin__stage_group_Unknown,clin__t_group_T1,clin__t_group_T2,clin__t_group_T3,clin__t_group_T4,clin__t_group_TX,clin__n_group_N0,clin__n_group_N1,clin__n_group_N2,clin__n_group_N3,clin__n_group_NX,clin__m_group_M0,clin__m_group_M1,clin__m_group_MX,clin__primary_diagnosis.diagnoses_Acinar cell carcinoma,clin__primary_diagnosis.diagnoses_Adenocarcinoma with mixed subtypes,"clin__primary_diagnosis.diagnoses_Adenocarcinoma, NOS","clin__primary_diagnosis.diagnoses_Bronchio-alveolar carcinoma, mucinous","clin__primary_diagnosis.diagnoses_Bronchiolo-alveolar adenocarcinoma, NOS","clin__primary_diagnosis.diagnoses_Bronchiolo-alveolar carcinoma, non-mucinous","clin__primary_diagnosis.diagnoses_Clear cell adenocarcinoma, NOS","clin__primary_diagnosis.diagnoses_Micropapillary carcinoma, NOS",clin__primary_diagnosis.diagnoses_Mucinous adenocarcinoma,"clin__primary_diagnosis.diagnoses_Papillary adenocarcinoma, NOS",clin__primary_diagnosis.diagnoses_Signet ring cell carcinoma,"clin__primary_diagnosis.diagnoses_Solid carcinoma, NOS",clin__morphology.diagnoses_8140/3,clin__morphology.diagnoses_8230/3,clin__morphology.diagnoses_8250/3,clin__morphology.diagnoses_8252/3,clin__morphology.diagnoses_8253/3,clin__morphology.diagnoses_8255/3,clin__morphology.diagnoses_8260/3,clin__morphology.diagnoses_8265/3,clin__morphology.diagnoses_8310/3,clin__morphology.diagnoses_8480/3,clin__morphology.diagnoses_8490/3,clin__morphology.diagnoses_8550/3,clin__icd_10_code.diagnoses_C34.0,clin__icd_10_code.diagnoses_C34.1,clin__icd_10_code.diagnoses_C34.2,clin__icd_10_code.diagnoses_C34.3,clin__icd_10_code.diagnoses_C34.30,clin__icd_10_code.diagnoses_C34.8,clin__icd_10_code.diagnoses_C34.9,"clin__tissue_or_organ_of_origin.diagnoses_Lower lobe, lung","clin__tissue_or_organ_of_origin.diagnoses_Lung, NOS",clin__tissue_or_organ_of_origin.diagnoses_Main bronchus,"clin__tissue_or_organ_of_origin.diagnoses_Middle lobe, lung",clin__tissue_or_organ_of_origin.diagnoses_Overlapping lesion of lung,"clin__tissue_or_organ_of_origin.diagnoses_Upper lobe, lung",DKK1,HMMR,CHEK1,CCR6,CDK1,GPI,BTK,SEMA3C,SEMA4B,AKR1A1,DDIT4,PPIA,KRAS,S100A10,PIM2,PSME3,CCL20,IKBKB,MIF,FGFR2,LTBR,MET,AGRN
0,TCGA-05-4249,1523.0,0,67.0,67.210959,52.0,2.849315,0,0,0,0,1,0,0.0,True,False,False,False,False,False,True,False,False,False,True,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,True,False,False,False,False,False,6.2236,6.5503,6.4044,8.8437,7.8490,12.6876,8.8659,8.8388,12.0981,11.1727,10.2499,11.5026,12.1734,12.2821,10.3286,11.6354,6.5895,12.2902,11.5286,9.1224,10.0792,13.1655,12.8819
1,TCGA-05-4250,121.0,1,79.0,79.638356,47.0,2.575342,0,0,0,0,0,0,0.0,False,False,True,False,False,False,False,True,False,False,False,True,False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,True,False,False,False,False,False,6.9698,9.4969,8.9563,5.1153,10.5075,13.2321,7.9330,12.5651,13.5400,10.6888,13.6731,12.9331,12.2059,14.6910,9.3912,12.0324,6.7385,10.1049,12.0569,6.6602,10.6575,13.0816,12.4818
2,TCGA-05-4382,607.0,0,68.0,68.131507,62.0,3.397260,0,0,0,0,1,1,0.0,True,False,False,False,False,False,True,False,False,False,True,False,False,False,F

In [6]:
print("Colunas principais:")
print(TIME_COL, "existe?", TIME_COL in df.columns)
print(EVENT_COL, "existe?", EVENT_COL in df.columns)

print("\nQuantidade de pacientes:", df.shape[0])
print("Quantidade de colunas:", df.shape[1])

Colunas principais:
surv__OS.time existe? True
surv__OS existe? True

Quantidade de pacientes: 502
Quantidade de colunas: 92


In [7]:
if "patient_id" in df.columns:
    patient_ids = df["patient_id"].copy()
    df = df.drop(columns=["patient_id"])
else:
    patient_ids = pd.Series(range(len(df)), name="patient_id")

print("Shape após remover patient_id:", df.shape)

Shape após remover patient_id: (502, 91)


In [8]:
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Total de nulos:", int(df.isna().sum().sum()))

nulos = df.isna().sum().sort_values(ascending=False)
display(nulos[nulos > 0].head(30))

Total de nulos: 1


clin__prior_treatment_bin    1
dtype: int64

In [9]:
# Não usar dropna geral.
# Se usar dropna geral, você pode perder pacientes sem perceber.

antes = len(df)

df = df.dropna(subset=[TIME_COL, EVENT_COL]).copy()

depois = len(df)

print("Removidos por nulo em tempo/evento:", antes - depois)

feature_cols_temp = [
    c for c in df.columns
    if c not in [TIME_COL, EVENT_COL]
]

for col in feature_cols_temp:
    if df[col].isna().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

print("Total de nulos após tratamento:", int(df.isna().sum().sum()))
print("Shape final:", df.shape)

Removidos por nulo em tempo/evento: 0
Total de nulos após tratamento: 0
Shape final: (502, 91)


In [10]:
df[EVENT_COL] = df[EVENT_COL].astype(int)
df[TIME_COL] = df[TIME_COL].astype(float)

print("Eventos:")
print(df[EVENT_COL].value_counts())

print("\nTaxa de eventos:", round(df[EVENT_COL].mean() * 100, 2), "%")

print("\nTempo de sobrevida:")
print("Mínimo:", df[TIME_COL].min())
print("Mediana:", df[TIME_COL].median())
print("Máximo:", df[TIME_COL].max())

Eventos:
surv__OS
0    320
1    182
Name: count, dtype: int64

Taxa de eventos: 36.25 %

Tempo de sobrevida:
Mínimo: 4.0
Mediana: 656.5
Máximo: 7248.0


In [11]:
feature_cols = [
    c for c in df.columns
    if c not in [TIME_COL, EVENT_COL]
]

# Garantia contra vazamento
feature_cols = [
    c for c in feature_cols
    if "vital_status" not in c.lower()
]

X = df[feature_cols].copy()
tempo = df[TIME_COL].copy()
evento = df[EVENT_COL].copy()

print("X:", X.shape)
print("tempo:", tempo.shape)
print("evento:", evento.shape)
print("Eventos:", int(evento.sum()))

X: (502, 89)
tempo: (502,)
evento: (502,)
Eventos: 182


In [12]:
skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

folds = list(skf.split(X, evento))

print("Quantidade de folds:", len(folds))

for i, (train_idx, test_idx) in enumerate(folds, start=1):
    print(
        f"Fold {i}:",
        "treino =", len(train_idx),
        "| teste =", len(test_idx),
        "| eventos teste =", int(evento.iloc[test_idx].sum())
    )

Quantidade de folds: 5
Fold 1: treino = 401 | teste = 101 | eventos teste = 37
Fold 2: treino = 401 | teste = 101 | eventos teste = 37
Fold 3: treino = 402 | teste = 100 | eventos teste = 36
Fold 4: treino = 402 | teste = 100 | eventos teste = 36
Fold 5: treino = 402 | teste = 100 | eventos teste = 36


## Pré-processamento dentro dos folds

A remoção de baixa variância, remoção de colinearidade e padronização serão feitas dentro de cada fold.

Isso evita que o conjunto de teste influencie o tratamento dos dados.

In [13]:
def tirar_baixa_variancia(X_train, X_test, threshold=0.01):
    seletor = VarianceThreshold(threshold=threshold)

    X_train_sel = seletor.fit_transform(X_train)
    X_test_sel = seletor.transform(X_test)

    colunas = X_train.columns[seletor.get_support()]

    X_train_sel = pd.DataFrame(
        X_train_sel,
        columns=colunas,
        index=X_train.index
    )

    X_test_sel = pd.DataFrame(
        X_test_sel,
        columns=colunas,
        index=X_test.index
    )

    return X_train_sel, X_test_sel

In [14]:
def tirar_colinearidade(X_train, X_test, limite=0.90):
    corr = X_train.corr().abs()

    matriz_superior = corr.where(
        np.triu(np.ones(corr.shape), k=1).astype(bool)
    )

    colunas_remover = [
        col for col in matriz_superior.columns
        if any(matriz_superior[col] > limite)
    ]

    X_train_ok = X_train.drop(columns=colunas_remover, errors="ignore")
    X_test_ok = X_test.drop(columns=colunas_remover, errors="ignore")

    return X_train_ok, X_test_ok, colunas_remover

In [15]:
def padronizar_dados(X_train, X_test):
    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    X_train_scaled = pd.DataFrame(
        X_train_scaled,
        columns=X_train.columns,
        index=X_train.index
    )

    X_test_scaled = pd.DataFrame(
        X_test_scaled,
        columns=X_test.columns,
        index=X_test.index
    )

    return X_train_scaled, X_test_scaled

In [16]:
def preparar_fold(X_train, X_test, usar_scaler=True, var_threshold=0.01, corr_threshold=0.90):
    # 1. tirar baixa variância
    X_train_prep, X_test_prep = tirar_baixa_variancia(
        X_train,
        X_test,
        threshold=var_threshold
    )

    # 2. tirar colunas muito correlacionadas
    X_train_prep, X_test_prep, removidas_corr = tirar_colinearidade(
        X_train_prep,
        X_test_prep,
        limite=corr_threshold
    )

    # 3. padronizar, se o modelo precisar
    if usar_scaler:
        X_train_prep, X_test_prep = padronizar_dados(
            X_train_prep,
            X_test_prep
        )

    return X_train_prep, X_test_prep, removidas_corr

In [17]:
def criar_y_sksurv(tempo_parte, evento_parte):
    return Surv.from_arrays(
        event=evento_parte.astype(bool).values,
        time=tempo_parte.astype(float).values
    )

In [18]:
resultados_folds = []
resumo_modelos = []

def guardar_resultado(modelo, fold, c_train, c_test):
    resultados_folds.append({
        "modelo": modelo,
        "fold": fold,
        "c_index_train": c_train,
        "c_index_test": c_test,
        "gap": c_train - c_test
    })


def criar_resumo_modelo(nome_modelo):
    temp = pd.DataFrame(resultados_folds)
    temp = temp[temp["modelo"] == nome_modelo]

    resumo = {
        "modelo": nome_modelo,
        "c_index_treino_medio": temp["c_index_train"].mean(),
        "c_index_teste_medio": temp["c_index_test"].mean(),
        "dp_teste": temp["c_index_test"].std(),
        "gap_medio": temp["gap"].mean()
    }

    resumo_modelos.append(resumo)

    return pd.DataFrame([resumo])

## Modelo 1 — CoxPH

O CoxPH será usado como modelo estatístico baseline.

Foi usada penalização para reduzir instabilidade dos coeficientes e diminuir risco de sobreajuste.

In [19]:
nome_modelo = "CoxPH"

for fold, (train_idx, test_idx) in enumerate(folds, start=1):
    print(f"\nRodando {nome_modelo} - Fold {fold}")

    X_train = X.iloc[train_idx].copy()
    X_test = X.iloc[test_idx].copy()

    tempo_train = tempo.iloc[train_idx]
    tempo_test = tempo.iloc[test_idx]

    evento_train = evento.iloc[train_idx]
    evento_test = evento.iloc[test_idx]

    X_train_prep, X_test_prep, removidas_corr = preparar_fold(
        X_train,
        X_test,
        usar_scaler=True,
        var_threshold=0.01,
        corr_threshold=0.90
    )

    df_train = X_train_prep.copy()
    df_train[TIME_COL] = tempo_train.values
    df_train[EVENT_COL] = evento_train.values

    cox = CoxPHFitter(penalizer=1.0)
    cox.fit(df_train, duration_col=TIME_COL, event_col=EVENT_COL)

    risco_train = cox.predict_partial_hazard(X_train_prep)
    risco_test = cox.predict_partial_hazard(X_test_prep)

    c_train = concordance_index(tempo_train, -risco_train, evento_train)
    c_test = concordance_index(tempo_test, -risco_test, evento_test)

    guardar_resultado(nome_modelo, fold, c_train, c_test)

    print("Features usadas:", X_train_prep.shape[1])
    print("C-index treino:", round(c_train, 4))
    print("C-index teste:", round(c_test, 4))


Rodando CoxPH - Fold 1
Features usadas: 57
C-index treino: 0.7362
C-index teste: 0.7771

Rodando CoxPH - Fold 2
Features usadas: 56
C-index treino: 0.7617
C-index teste: 0.6579

Rodando CoxPH - Fold 3
Features usadas: 57
C-index treino: 0.7424
C-index teste: 0.7463

Rodando CoxPH - Fold 4
Features usadas: 56
C-index treino: 0.752
C-index teste: 0.7098

Rodando CoxPH - Fold 5
Features usadas: 57
C-index treino: 0.7524
C-index teste: 0.6883


In [20]:
resumo_cox = criar_resumo_modelo("CoxPH")
display(resumo_cox)

,modelo,c_index_treino_medio,c_index_teste_medio,dp_teste,gap_medio
0,CoxPH,0.748936,0.715851,0.046994,0.033085


## Modelo 2 — Weibull AFT

O Weibull AFT é um modelo paramétrico de sobrevida.

Para LUAD, será usado `penalizer=0.1`, conforme os testes feitos anteriormente.

In [21]:
nome_modelo = "Weibull AFT"

for fold, (train_idx, test_idx) in enumerate(folds, start=1):
    print(f"\nRodando {nome_modelo} - Fold {fold}")

    X_train = X.iloc[train_idx].copy()
    X_test = X.iloc[test_idx].copy()

    tempo_train = tempo.iloc[train_idx]
    tempo_test = tempo.iloc[test_idx]

    evento_train = evento.iloc[train_idx]
    evento_test = evento.iloc[test_idx]

    X_train_prep, X_test_prep, removidas_corr = preparar_fold(
        X_train,
        X_test,
        usar_scaler=True,
        var_threshold=0.01,
        corr_threshold=0.90
    )

    df_train = X_train_prep.copy()
    df_train[TIME_COL] = tempo_train.values
    df_train[EVENT_COL] = evento_train.values

    aft = WeibullAFTFitter(penalizer=0.1)
    aft.fit(df_train, duration_col=TIME_COL, event_col=EVENT_COL)

    pred_train = aft.predict_median(X_train_prep)
    pred_test = aft.predict_median(X_test_prep)

    pred_train = pred_train.replace([np.inf, -np.inf], np.nan)
    pred_test = pred_test.replace([np.inf, -np.inf], np.nan)

    pred_train = pred_train.fillna(pred_train.max())
    pred_test = pred_test.fillna(pred_test.max())

    c_train = concordance_index(tempo_train, pred_train, evento_train)
    c_test = concordance_index(tempo_test, pred_test, evento_test)

    guardar_resultado(nome_modelo, fold, c_train, c_test)

    print("Features usadas:", X_train_prep.shape[1])
    print("C-index treino:", round(c_train, 4))
    print("C-index teste:", round(c_test, 4))


Rodando Weibull AFT - Fold 1
Features usadas: 57
C-index treino: 0.755
C-index teste: 0.7592

Rodando Weibull AFT - Fold 2
Features usadas: 56
C-index treino: 0.7871
C-index teste: 0.6425

Rodando Weibull AFT - Fold 3
Features usadas: 57
C-index treino: 0.7682
C-index teste: 0.7341

Rodando Weibull AFT - Fold 4
Features usadas: 56
C-index treino: 0.7755
C-index teste: 0.6967

Rodando Weibull AFT - Fold 5
Features usadas: 57
C-index treino: 0.782
C-index teste: 0.6795


In [22]:
resumo_weibull = criar_resumo_modelo("Weibull AFT")
display(resumo_weibull)

,modelo,c_index_treino_medio,c_index_teste_medio,dp_teste,gap_medio
0,Weibull AFT,0.773544,0.702402,0.045754,0.071143


## Modelo 3 — Random Survival Forest

O RSF é uma adaptação da Random Forest para análise de sobrevida.

Como é baseado em árvores, não precisa de `StandardScaler`.

In [23]:
nome_modelo = "RSF"

for fold, (train_idx, test_idx) in enumerate(folds, start=1):
    print(f"\nRodando {nome_modelo} - Fold {fold}")

    X_train = X.iloc[train_idx].copy()
    X_test = X.iloc[test_idx].copy()

    tempo_train = tempo.iloc[train_idx]
    tempo_test = tempo.iloc[test_idx]

    evento_train = evento.iloc[train_idx]
    evento_test = evento.iloc[test_idx]

    X_train_prep, X_test_prep, removidas_corr = preparar_fold(
        X_train,
        X_test,
        usar_scaler=False,
        var_threshold=0.01,
        corr_threshold=0.90
    )

    y_train = criar_y_sksurv(tempo_train, evento_train)

    rsf = RandomSurvivalForest(
        n_estimators=200,
        min_samples_split=25,
        min_samples_leaf=20,
        max_features=5,
        n_jobs=-1,
        random_state=SEED
    )

    rsf.fit(X_train_prep, y_train)

    risco_train = rsf.predict(X_train_prep)
    risco_test = rsf.predict(X_test_prep)

    c_train = concordance_index_censored(
        evento_train.astype(bool),
        tempo_train,
        risco_train
    )[0]

    c_test = concordance_index_censored(
        evento_test.astype(bool),
        tempo_test,
        risco_test
    )[0]

    guardar_resultado(nome_modelo, fold, c_train, c_test)

    print("Features usadas:", X_train_prep.shape[1])
    print("C-index treino:", round(c_train, 4))
    print("C-index teste:", round(c_test, 4))


Rodando RSF - Fold 1
Features usadas: 57
C-index treino: 0.8038
C-index teste: 0.7249

Rodando RSF - Fold 2
Features usadas: 56
C-index treino: 0.8202
C-index teste: 0.6482

Rodando RSF - Fold 3
Features usadas: 57
C-index treino: 0.8039
C-index teste: 0.7256

Rodando RSF - Fold 4
Features usadas: 56
C-index treino: 0.8054
C-index teste: 0.7261

Rodando RSF - Fold 5
Features usadas: 57
C-index treino: 0.8103
C-index teste: 0.6785


In [24]:
resumo_rsf = criar_resumo_modelo("RSF")
display(resumo_rsf)

,modelo,c_index_treino_medio,c_index_teste_medio,dp_teste,gap_medio
0,RSF,0.808696,0.700668,0.035689,0.108028


## Modelo 4 — Gradient Boosting Survival Analysis

O GBSA é um modelo baseado em boosting.

Ele cria árvores de forma sequencial, tentando corrigir os erros do modelo anterior.

Como também é baseado em árvores, não precisa de padronização.

In [25]:
nome_modelo = "GBSA"

for fold, (train_idx, test_idx) in enumerate(folds, start=1):
    print(f"\nRodando {nome_modelo} - Fold {fold}")

    X_train = X.iloc[train_idx].copy()
    X_test = X.iloc[test_idx].copy()

    tempo_train = tempo.iloc[train_idx]
    tempo_test = tempo.iloc[test_idx]

    evento_train = evento.iloc[train_idx]
    evento_test = evento.iloc[test_idx]

    X_train_prep, X_test_prep, removidas_corr = preparar_fold(
        X_train,
        X_test,
        usar_scaler=False,
        var_threshold=0.01,
        corr_threshold=0.90
    )

    y_train = criar_y_sksurv(tempo_train, evento_train)

    gbsa = GradientBoostingSurvivalAnalysis(
        n_estimators=150,
        learning_rate=0.02,
        max_depth=1,
        subsample=0.7,
        random_state=SEED
    )

    gbsa.fit(X_train_prep, y_train)

    risco_train = gbsa.predict(X_train_prep)
    risco_test = gbsa.predict(X_test_prep)

    c_train = concordance_index_censored(
        evento_train.astype(bool),
        tempo_train,
        risco_train
    )[0]

    c_test = concordance_index_censored(
        evento_test.astype(bool),
        tempo_test,
        risco_test
    )[0]

    guardar_resultado(nome_modelo, fold, c_train, c_test)

    print("Features usadas:", X_train_prep.shape[1])
    print("C-index treino:", round(c_train, 4))
    print("C-index teste:", round(c_test, 4))


Rodando GBSA - Fold 1
Features usadas: 57
C-index treino: 0.7456
C-index teste: 0.6961

Rodando GBSA - Fold 2
Features usadas: 56
C-index treino: 0.766
C-index teste: 0.6819

Rodando GBSA - Fold 3
Features usadas: 57
C-index treino: 0.7406
C-index teste: 0.7665

Rodando GBSA - Fold 4
Features usadas: 56
C-index treino: 0.7515
C-index teste: 0.7196

Rodando GBSA - Fold 5
Features usadas: 57
C-index treino: 0.7632
C-index teste: 0.6865


In [26]:
resumo_gbsa = criar_resumo_modelo("GBSA")
display(resumo_gbsa)

,modelo,c_index_treino_medio,c_index_teste_medio,dp_teste,gap_medio
0,GBSA,0.753377,0.7101,0.034696,0.043277


## Modelo 5 — DeepSurv

O DeepSurv é uma rede neural baseada na função de risco do Cox.

Para LUAD, será usada uma rede menor, com camadas `[16, 8]`, para reduzir sobreajuste.

In [27]:
def preparar_dados_deep(X_train, X_test):
    X_train_deep = X_train.astype("float32").values
    X_test_deep = X_test.astype("float32").values

    return X_train_deep, X_test_deep

In [28]:
if DEEP_OK:
    nome_modelo = "DeepSurv"

    for fold, (train_idx, test_idx) in enumerate(folds, start=1):
        print(f"\nRodando {nome_modelo} - Fold {fold}")

        X_train = X.iloc[train_idx].copy()
        X_test = X.iloc[test_idx].copy()

        tempo_train = tempo.iloc[train_idx].astype("float32")
        tempo_test = tempo.iloc[test_idx].astype("float32")

        evento_train = evento.iloc[train_idx].astype("float32")
        evento_test = evento.iloc[test_idx].astype("float32")

        X_train_prep, X_test_prep, removidas_corr = preparar_fold(
            X_train,
            X_test,
            usar_scaler=True,
            var_threshold=0.01,
            corr_threshold=0.90
        )

        x_train_deep, x_test_deep = preparar_dados_deep(
            X_train_prep,
            X_test_prep
        )

        in_features = x_train_deep.shape[1]

        net = tt.practical.MLPVanilla(
            in_features=in_features,
            num_nodes=[16, 8],
            out_features=1,
            batch_norm=True,
            dropout=0.1
        )

        model = CoxPH(
            net,
            tt.optim.Adam
        )

        model.optimizer.set_lr(0.001)

        callbacks = [
            tt.callbacks.EarlyStopping(patience=10)
        ]

        model.fit(
            x_train_deep,
            (tempo_train.values, evento_train.values),
            batch_size=32,
            epochs=100,
            callbacks=callbacks,
            val_data=(x_test_deep, (tempo_test.values, evento_test.values)),
            verbose=False
        )

        model.compute_baseline_hazards(
            input=x_train_deep,
            target=(tempo_train.values, evento_train.values)
        )

        surv_train = model.predict_surv_df(x_train_deep)
        surv_test = model.predict_surv_df(x_test_deep)

        ev_train = EvalSurv(
            surv_train,
            tempo_train.values,
            evento_train.values,
            censor_surv="km"
        )

        ev_test = EvalSurv(
            surv_test,
            tempo_test.values,
            evento_test.values,
            censor_surv="km"
        )

        c_train = ev_train.concordance_td("antolini")
        c_test = ev_test.concordance_td("antolini")

        guardar_resultado(nome_modelo, fold, c_train, c_test)

        print("Features usadas:", X_train_prep.shape[1])
        print("C-index treino:", round(c_train, 4))
        print("C-index teste:", round(c_test, 4))

else:
    print("DeepSurv não será executado porque pycox/torch não carregou.")


Rodando DeepSurv - Fold 1
Features usadas: 57
C-index treino: 0.7786
C-index teste: 0.7036

Rodando DeepSurv - Fold 2
Features usadas: 56
C-index treino: 0.813
C-index teste: 0.5939

Rodando DeepSurv - Fold 3
Features usadas: 57
C-index treino: 0.7761
C-index teste: 0.6937

Rodando DeepSurv - Fold 4
Features usadas: 56
C-index treino: 0.7942
C-index teste: 0.7125

Rodando DeepSurv - Fold 5
Features usadas: 57
C-index treino: 0.8449
C-index teste: 0.6929
